In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchao

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

#Hugging Face classes chosen to handle pre-trained LLMs and abstract away complex training loops
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
#Parameter-Efficient Fine-Tuning libraries.
from peft import get_peft_model, LoraConfig, TaskType
import wandb
from kaggle_secrets import UserSecretsClient

In [2]:
# All files under the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# SETUP & CONFIGURATION

In [3]:
seed=42
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Global constants
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WandB-API")

wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

# EDA

In [5]:
TRAIN_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

In [6]:
TRAIN_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [7]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [8]:
TRAIN_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


.describe(include='all') generates descriptive statistics. Using include='all' ensures it also summarizes object/string columns showing counts, unique values, and the most frequent value.

.info() prints a concise summary of the DataFrame, including the index dtype and columns, non-null values, and memory usage.


# DATA PREPROCESSING

## Check Null Values

In [9]:
mcq_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

TRAIN_df[mcq_cols].isna().sum()
#So there's no null value in this dataset

prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

## Drop rows with duplicate prompt

In [10]:
TRAIN_df['prompt'].duplicated().sum()

np.int64(242)

In [11]:
TRAIN_df[TRAIN_df['prompt'].duplicated(keep=False)].sort_values('prompt').head(3)

,id,prompt,A,B,C,D,E,answer
605,606,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
456,457,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E
439,440,Choose the correct answer: What are permutatio...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,Permutation-inversion groups are groups of sym...,E


Keep the first occurrence of each duplicate prompt and drops the rest:

In [12]:
TRAIN_df = TRAIN_df.drop_duplicates(subset='prompt', keep='first')
TRAIN_df['prompt'].duplicated().sum()  # should now be 0

np.int64(0)

## Case normalization - Convert all the text to lowercase

In [13]:
TRAIN_df[mcq_cols] = TRAIN_df[mcq_cols].apply(lambda col: col.str.lower())
test_df[mcq_cols] = test_df[mcq_cols].apply(lambda col: col.str.lower())

In [14]:
# Display an example
print(TRAIN_df['prompt'].iloc[0])

pick the best possible answer: what is martin heidegger's view on the relationship between time and human existence? among the listed options.


## Trip possible boiler plates

In [15]:
import re
def strip_boilerplate(text):
    return re.sub(r'^(pick the best possible answer|select the most accurate option|determine the correct option|choose the correct answer|identify the correct statement)\s*:\s*', '', text)

TRAIN_df['prompt'] = TRAIN_df['prompt'].apply(strip_boilerplate)
test_df['prompt'] = test_df['prompt'].apply(strip_boilerplate)

## Encode Options (answers)

In [16]:
encoder = LabelEncoder()
y = TRAIN_df['answer']
y = encoder.fit_transform(y)

In [17]:
TRAIN_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
1,2,what is accelerator-based light-ion fusion?,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,A
2,3,what is the term used in astrophysics to descr...,blueshifting,redshifting,reddening,whitening,yellowing,C
3,4,what is martin heidegger's view on the relatio...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,B
4,5,what is the concept of simultaneity in einstei...,"simultaneity is relative, meaning that two eve...","simultaneity is relative, meaning that two eve...","simultaneity is absolute, meaning that two eve...",simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,A


## Split Train and Validation Set

`stratify` is chosen over a random split to guarantee that both datasets contain the exact same proportion of A, B, C, D, and E answers, preventing class imbalance

In [18]:
# Stratified split to ensure answer distributions match
train_df, val_df = train_test_split(TRAIN_df, test_size=0.2, random_state=42, stratify=TRAIN_df['answer'])

In [19]:
train_df.shape[0]

1406

In [20]:
val_df.shape[0]

352

# METRIC EVALUATION FUNCTION

It extracts logits (raw model predictions) and labels, and calculates standard Accuracy, Macro F1-score, and iteratively calculates the MAP@3 score.

In [21]:
def calculate_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argsort(logits, axis=-1)[:, ::-1] # Sort in descending order

    top1_preds = preds[:, 0]
    accuracy = accuracy_score(labels, top1_preds)
    f1 = f1_score(labels, top1_preds, average="macro")

    map3_score = 0.0
    for i in range(len(labels)):
        true_label = labels[i]
        for rank in range(3):
            if preds[i, rank] == true_label:
                map3_score += 1.0 / (rank + 1)
                break
    map3_score /= len(labels)
    
    return {
        "accuracy": accuracy,
        "f1_score": f1,
        "map@3": map3_score
    }

`argsort` is chosen over `max` because we need to calculate Mean Average Precision at 3 (MAP@3) which requires top 3 rankings, not just the single best answer.

# HUGGING FACE TRANSFORMER DATASETS

`tokenizer` - Converts raw text into token IDs.

Padding and truncation are chosen over dynamic lengths so the outputs form perfect rectangular tensors [Batch_Size, Num_Choices, Max_Length] that PyTorch can process on the GPU.

In [22]:
class HuggingFaceMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        choices_inputs = []
        for opt in self.options:
            option_text = str(row[opt])
            choices_inputs.append((prompt, option_text))
            
        # Standard tokenization structure matching [Batch_Size, Num_Choices, Max_Length]
        features = self.tokenizer(
            [text[0] for text in choices_inputs],
            [text[1] for text in choices_inputs],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": features["input_ids"],
            "attention_mask": features["attention_mask"]
        }
        
        if not self.is_test:
            item["labels"] = torch.tensor(LABEL_MAP[row['answer']], dtype=torch.long)
            
        return item

# Data collator to enforce precise shapes for Hugging Face multi-choice pipeline execution
def mcq_data_collator(features):
    batch = {}
    batch["input_ids"] = torch.stack([f["input_ids"] for f in features])
    batch["attention_mask"] = torch.stack([f["attention_mask"] for f in features])
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch

`mcq_data_collator` - Takes a list of individual dictionary items and uses torch.stack to stack them into single batched tensors

# Model: PRETRAINED DeBERTa

**Decoding-enhanced BERT with Disentangled Attention**

Instead of treating words as single vectors, DeBERTa treats words as two separate vectors representing their content and position. This significantly improves how the model understands the context and relationship of words in a sentence.

In [23]:
deberta_ckpt = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_ckpt)
deberta_model = AutoModelForMultipleChoice.from_pretrained(deberta_ckpt).to(DEVICE)
deberta_train_ds = HuggingFaceMCQDataset(train_df, deberta_tokenizer, max_len=128)
deberta_val_ds = HuggingFaceMCQDataset(val_df, deberta_tokenizer, max_len=128)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                 

In [24]:
lens = deberta_tokenizer(train_df['prompt'].tolist(), truncation=False)['input_ids']
print(np.percentile([len(l) for l in lens], [50, 90, 95, 99]))

[18. 30. 34. 44.]


99th percentile prompt is 45 tokens so it is safe to use max_len=128

In [25]:
NUM_EPOCHS = 4

deberta_args = TrainingArguments(
    output_dir="./deberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-6,                
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,  #Fraction of total training steps spent increasing the lr from 0 to the initial value                
    max_grad_norm=0.5,
    per_device_train_batch_size=8,      
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   #calculates gradients for 2 smaller batches before updating weights (Effective batch size = 8 × 2 × 2 = 32)
    num_train_epochs=NUM_EPOCHS,
    metric_for_best_model='map@3',
    load_best_model_at_end=True,
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name="pretrained-deberta-run",
    logging_steps=10,
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_args,
    train_dataset=deberta_train_ds,
    eval_dataset=deberta_val_ds,
    data_collator=mcq_data_collator,
    compute_metrics=calculate_metrics,
)
deberta_trainer.train()
wandb.finish()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Score,Map@3
1,4.910352,1.752930,0.818182,0.815920,0.883049
2,2.869922,0.861816,0.923295,0.919122,0.952178
3,0.710509,0.194092,0.997159,0.997158,0.998580
4,0.201736,0.076355,0.997159,0.997158,0.998580


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

eval/accuracy,▁▅██
eval/f1_score,▁▅██
eval/loss,█▄▁▁
eval/map@3,▁▅██
eval/runtime,▁█▅▆
eval/samples_per_second,█▁▄▃
eval/steps_per_second,█▁▄▃
train/epoch,▁▁▂▂▂▃▃▄▄▄▄▅▅▆▆▆▆▇▇███
train/global_step,▁▁▂▂▂▃▃▄▄▄▄▅▅▆▆▆▆▇▇███
train/grad_norm,▃▃▄▆█▆▅█▃▃▂▂▁▁▁▃▂
+2,...


In [26]:
# PREDICTIONS FOR DeBERTa
# Create the test dataset and dataloader for DeBERTa
test_deberta_ds = HuggingFaceMCQDataset(test_df, deberta_tokenizer, max_len=128, is_test=True)
deberta_test_loader = DataLoader(test_deberta_ds, batch_size=4, shuffle=False, collate_fn=mcq_data_collator)

# Set the model to evaluation mode
deberta_model.eval()
deberta_probs = []

# Run inference without calculating gradients
with torch.no_grad():
    for batch in deberta_test_loader:
        inputs = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = deberta_model(**inputs).logits
        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)
        deberta_probs.append(probs.cpu().numpy())
        
# Concatenate all batches into a single numpy array
deberta_probs = np.concatenate(deberta_probs, axis=0)

# 4. Extract Top-3 space-separated string maps for output submissions
deberta_submission_predictions = []
for probs in deberta_probs:
    # Sort indices in descending order based on probability and grab the top 3
    top3_indices = np.argsort(probs)[::-1][:3]
    # Map numeric indices back to 'A', 'B', 'C', 'D', 'E'
    top3_labels = [INV_LABEL_MAP[idx] for idx in top3_indices]
    deberta_submission_predictions.append(" ".join(top3_labels))

# Kaggle submission
submission_df2 = pd.DataFrame({
    "id": test_df["id"],
    "Prediction": deberta_submission_predictions
})

In [27]:
submission_df2.to_csv("submission.csv", index=False)
print("Inference completed successfully. Output saved to: submission.csv")

Inference completed successfully. Output saved to: submission.csv
